# NB07 — Honest Backtesting
**Machine Learning in Finance · Week 7**  [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ruiyuan-chen/ml-finance-course/blob/main/notebooks/NB07_honest_backtesting.ipynb)

**Finance question:** My model picks winners a bit better than chance. Will it make money?

**Key idea:** A **walk-forward backtest** replays history the way an investor lives it: re-train on the past, trade the next year, repeat. Then the real-world bills arrive: **trading costs**, **turnover** and **luck**.

In [ ]:
# ▶ RUN ME FIRST — loads the course tools and data (about 30 seconds the first time)
COURSE_URL = "https://raw.githubusercontent.com/ruiyuan-chen/ml-finance-course/main/"
import os, sys, urllib.request
here = [f for f in [".", "..", "../.."] if os.path.exists(os.path.join(f, "mlfin.py"))]
if here:
    sys.path.insert(0, here[0])
else:
    urllib.request.urlretrieve(COURSE_URL + "mlfin.py", "mlfin.py")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import mlfin
mlfin.setup(COURSE_URL)

## Step 1: the walk-forward loop
Every January the model is re-trained on everything known so far, then it picks stocks for that year
only. It never sees the year it trades. This is the course backtest, `mlfin.walk_forward_backtest`.

In [ ]:
mlfin.plot_backtest_loop(test_start=2015, end=2025)

## Step 2: run the course model as a fund
The model is the course default from NB06 (boosting, predicting "beats SPY next month"). Each rebalance
date it buys the stocks with the highest predicted probability. Four dials: how many stocks, how often
you trade, what each trade costs, and how you weight the stocks.

In [ ]:
# 🟡 YOUR TURN
N_LONG = 30          # ← how many stocks to hold: try 10 or 100
REBALANCE = "M"      # ← "M" monthly, "Q" quarterly, "A" once a year
COST_BPS = 20        # ← cost per dollar traded, in basis points (20 bps = 0.20%)
WEIGHTING = "equal"  # ← "equal", "rank" (more in the top picks) or "inverse_vol" (less in jumpy stocks)

panel = mlfin.make_features()
result = mlfin.walk_forward_backtest(panel, mlfin.default_model(), n_long=N_LONG, rebalance=REBALANCE,
                                     cost_bps=COST_BPS, weighting=WEIGHTING)
result["stats"]

*How to read the scorecard:* **Ann. return** = average yearly growth. **Volatility** = how much it swings.
**Sharpe ratio** = return above cash per unit of swing (higher is better). **Max drawdown** = worst fall
from a peak. **Turnover/yr** = how much of the portfolio you replace each year (8 = eight times).

In [ ]:
mlfin.plot_cumulative(result)
mlfin.plot_drawdown(result)

## Step 3: the bill for trading
The picks don't change with costs; only what you pay to make them. One backtest, re-priced at
0, 10, 20 and 50 basis points per dollar traded.

In [ ]:
mlfin.cost_experiment(result, costs=[0, 10, 20, 50])

*What to look for:* the gap between the top line (free trading) and the others. A strategy that
replaces its whole portfolio eight times a year pays the cost eight times a year.

## Step 4: trade less often?
Rebalancing monthly uses the freshest predictions but trades a lot. Quarterly or yearly trades less.
Which wins after costs? (This runs three backtests: about 15 seconds.)

In [ ]:
mlfin.rebalance_experiment(panel, mlfin.default_model(), freqs=["M", "Q", "A"], cost_bps=COST_BPS)

## Step 5: so how "accurate" is it?
The finance question was about a model that is "55% accurate". Here is ours: of the stocks it bought,
how many beat SPY the next month, compared with all stocks?

In [ ]:
mlfin.pick_hit_rate(result, panel)

## Step 6: skill or luck?
Even a real edge is small next to the month-to-month noise of stock returns. The **t-statistic** asks
how many "standard errors" the average edge is away from zero. Below about 2, luck explains it easily.

In [ ]:
mlfin.excess_return_tstat(result)

## Step 7: faster signals, bigger bills
Short-term features (last month's return `rev_1`, the biggest one-day jump `max_1`) change every month,
so a model that uses them trades more. Add them and compare turnover and return after costs.

In [ ]:
# 🟡 YOUR TURN
EXTRA = ["rev_1", "max_1"]   # ← try [] (none), ["rev_1"], or other features from mlfin.FEATURE_DESCRIPTIONS

fast = mlfin.walk_forward_backtest(panel, mlfin.default_model(), features=mlfin.DEFAULT_BACKTEST_FEATURES + EXTRA,
                                   n_long=N_LONG, rebalance=REBALANCE, cost_bps=COST_BPS, verbose=False)
pd.DataFrame({"default features": result["stats"]["Strategy (after costs)"],
              "with extra features": fast["stats"]["Strategy (after costs)"]})

## ⭐ GOING FURTHER (optional)
1. **Luck in the machine.** The course model has a random element. Replace `mlfin.default_model()` in Step 2
   with `HistGradientBoostingClassifier(max_iter=200, learning_rate=0.05, max_depth=3, random_state=1)`
   (import it from `sklearn.ensemble`) and try seeds 1–4. How much does the annual return move?
2. Try `WEIGHTING = "inverse_vol"`. Does the drawdown shrink? What does it cost in return?
3. Ask an AI assistant: *"Why can a strategy with a higher return than SPY still be a worse investment?"*

## 🪞 REFLECTION (graded — write in your own words, no AI)

Double-click this cell and replace the *...* with your answers.

1. **In one sentence, what finance question did this week's technique answer?**
   *...*
2. **What is one way the result could be misleading?**
   *...*
3. **What did you ask an AI assistant this week, and was its answer right?**
   *...*